In [1]:
!pip install --quiet \
langchain==0.2.16 \
langchain-community==0.2.9 \
langchain-core==0.2.38 \
langchain-google-genai==1.0.8 \
google-generativeai \
numpy==1.26.4 \
scikit-learn==1.5.2 \
matplotlib==3.10.0 \
fsspec==2025.3.0 \
langchain_experimental \
langchain_community \
chromadb \
pydantic \
--upgrade --no-cache-dir

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 5.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 27.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 155.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.4/396.4 kB 277.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 209.5 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 220.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 138.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.2/164.2 kB 205.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 718.3/718.3 kB 237.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.2/203.2 kB 222.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
import os 
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
os.environ["GOOGLE_API_KEY"] = user_secrets.get_secret("Google_API_Key")
os.environ["HUGGINGFACEHUB_API_TOKEN"] = user_secrets.get_secret("HUGGINGFACEHUB_API_TOKEN")
os.environ["ExchangeRateAPI"] = user_secrets.get_secret("ExchangeRateAPI")

136f22feef3a81a66dd09501


# **Chat Model**

In [76]:
from langchain.chat_models import init_chat_model

In [77]:
# Initialize gemini gen-ai model 
model = init_chat_model(model='gemini-2.0-flash', model_provider='google-genai')

In [78]:
# get model response

result = model.invoke('Hi, Im Harsh').content
print(result)

Hi Harsh, it's nice to meet you! How can I help you today?



# **Embedding Model**

In [79]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [80]:
# Initialize the Google Generative AI embeddings model
google_embeddings = GoogleGenerativeAIEmbeddings(
    model='models/gemini-embedding-exp-03-07',  # Specify the model name
)

In [81]:
# The embeddings model is now ready to be used in your application
query = "What is the capital of UP?"

# Generate embedding for the query using Google GenAI
query_embedding = google_embeddings.embed_query(query)

In [82]:
from sklearn.metrics.pairwise import cosine_similarity

documents = [
    "The capital of France is Paris.",
    "The capital of Germany is Berlin.",
    "The capital of Italy is Rome.",
    "The capital of Spain is Madrid.",
]

query = "What is the capital of France?"

# Generate embeddings for the documents using Google GenAI
google_document_embeddings = google_embeddings.embed_documents(documents)
# Generate embedding for the query using Google GenAI
google_query_embedding = google_embeddings.embed_query(query)

# Calculate cosine similarity between the query embedding and document embeddings using Google GenAI
google_similarities = cosine_similarity([google_query_embedding], google_document_embeddings)[0]

print(google_similarities)  # Output the cosine similarity scores from Google GenAI

[0.78928934 0.66698179 0.67434955 0.66744517]


# **Prompting**

## **Prompt Template**

In [83]:
from langchain.prompts import PromptTemplate

In [84]:
# Initialize the chat model
model = init_chat_model(model='gemini-2.0-flash', model_provider='google-genai')

# Define a prompt template
prompt_template = PromptTemplate(
    input_variables=["name"],   # Specify the input variable            
    template="Hello, I'm {name}!"  # Define the template
)

In [85]:
# Get prompt

prompt = prompt_template.invoke(
    input={
        'name':'Harsh'
    }
)
print(prompt.text)

Hello, I'm Harsh!


In [86]:
# Get result using template

result = model.invoke(prompt_template.format(name='Harsh'))
print(result.content)

Hello Harsh! It's nice to meet you. How can I help you today?



## **ChatPromptTemplate**

In [87]:
from langchain.prompts import ChatPromptTemplate

In [88]:
# Initialize the chat model
model = init_chat_model(model='gemini-2.0-flash', model_provider='google-genai')

In [89]:
# initialize ChatPrompt
chat_template = ChatPromptTemplate(
    [
        ("system", "You are a helpful weather assistant that provides information about the {topic}."),
        ("human", "who are you?")
    ]
)

prompt = chat_template.invoke(
    input={
        "topic": "weather"
    }
)
print(prompt)

messages=[SystemMessage(content='You are a helpful weather assistant that provides information about the weather.'), HumanMessage(content='who are you?')]


In [90]:
# Get result according to chatprompttemplate
model_response = model.invoke(prompt)

# Output the content of the response
print(model_response.content)  # Should print the model's response to the weather query

I am a weather assistant. I can provide you with current weather conditions, forecasts, and other weather-related information. Just let me know what you need!



## **Message Placeholder**

In [91]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.schema import HumanMessage, AIMessage

In [92]:
# Define the prompt template 
chat_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that provides information about the {topic}."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{human_input}"),
])

In [93]:
# Define proper chat history using message classes
chat_history = [
    HumanMessage(content="What is status of my refund of order id 10001?"),
    AIMessage(content="Your order has been delivered by 10th of Agust."),
    HumanMessage(content="What is the status of my refund of order id 10002?"),
    AIMessage(content="Your order will be delivered by 15th of Agust.")
]

In [94]:
# Initialize the model (ensure your environment has the right credentials set)
model = init_chat_model(model='gemini-2.0-flash', model_provider='google-genai')

In [95]:
# Create the prompt
prompt = chat_template.invoke(
    {
        "topic": "refund status",
        "human_input": "What is the status of my refund of order id 10001?",
        "chat_history": chat_history
    }
)

# Generate the response
response = model.invoke(prompt)

# Output the model response
print(response.content)

Your refund for order ID 10001 is currently being processed. You should receive the refund within 5-7 business days.



## **Zero Shot Prompting**

In [96]:
from langchain_core.prompts import ChatPromptTemplate

In [97]:
# Initialize the chat model
model = init_chat_model(model='gemini-2.0-flash', model_provider='google-genai')

In [98]:
# Define Examples as (role, message) tuple

examples = [
    ('human', 'Can i have 1 coldrink'),
    ('ai', '1'),
    ('human', 'I want to have 5 burgers'),
    ('ai', '5'),
    ('human', 'Can i get 10 samosa'),
    ('ai', '10')
]

In [99]:
# define input
input_text = 'Can i have 10 pizza'

In [100]:
# Build the full prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that takes food orders and returns number of items."),
    *examples,
    ("human", "{input}")
])

In [101]:
# Create the chain
chain = prompt | model

In [102]:
# Run the model
response = chain.invoke({
    "input": input_text
})

print(response.content)

10


## **Few Shot Prompting**

In [103]:
from langchain_core.prompts import ChatPromptTemplate

In [104]:
# Initialize the chat model
model = init_chat_model(model='gemini-2.0-flash', model_provider='google-genai')

In [105]:
# Define the examples as (role, message) tuples
examples = [
    ("human", "Can i have a pizza with extra cheese?"),
    ("ai", '''{{
        "items": [
            {{
                "item": "pizza",
                "quantity": 1,
                "type": "extra cheese",
                "toppings": ["mushroom", "olive"]
            }}
        ]
    }}'''),

    ("human", "Can i have 2 water bottle and a pizza?"),
    ("ai", '''{{
        "items": [
            {{
                "item": "pizza",
                "quantity": 1,
                "type": "regular",
                "toppings": ["mushroom", "olive"]
            }},
            {{
                "item": "water_bottle",
                "quantity": 2
            }}
        ]
    }}'''),

    ("human", "Can i have 5 chips and 2 burgers?"),
    ("ai", '''{{
        "items": [
            {{
                "item": "burger",
                "quantity": 2,
                "type": "regular",
                "toppings": ["mushroom", "olive"]
            }},
            {{
                "item": "chips",
                "quantity": 5
            }}
        ]
    }}''')
]

In [106]:
# Define input
input_text = "Can i have 5 banana and 1 pizza?"

In [107]:
# Build the full prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that takes food orders and returns them in JSON format."),
    *examples,
    ("human", "{input}")
])

In [108]:
# Create the chain
chain = prompt | model

# Run the model
response = chain.invoke({
    "input": input_text
})

print(response.content)

```json
{
    "items": [
        {
            "item": "banana",
            "quantity": 5
        },
        {
            "item": "pizza",
            "quantity": 1,
            "type": "regular",
            "toppings": []
        }
    ]
}
```


# **Structured Output**

In [109]:
from typing import Optional
from typing_extensions import Annotated, TypedDict
from pydantic import BaseModel, Field
from langchain.prompts import PromptTemplate

In [110]:
# define structure for output

class Joke(BaseModel):
    setup: str
    punchline: str
    key: list[str] = Field(description='key points of joke')
    rating: float = Field(description='rate joke between 0 to 5')

In [111]:
# define structured model
model = init_chat_model("gemini-2.0-flash", model_provider="google-genai")
strutured_model = model.with_structured_output(Joke)

In [112]:
# invoke structured model
response = strutured_model.invoke("i see a  ant on the ground, it is carrying a leaf")
print(response)

[]


In [113]:
# initialize template
template = PromptTemplate(
    input_variables = ["topic"], 
    template='tell me a joke on {topic}'
)

In [114]:
# create a chain 
chain = template | strutured_model 

# invoke chain
response = chain.invoke({
    "topic" : "aunt"
})

print(response)

[]


# **Output Parser**

## **Str Output Parser**

In [115]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [116]:
# initialize chat model
model = init_chat_model("gemini-2.0-flash", model_provider="google-genai")

In [117]:
# initialize a prompt template
prompt = PromptTemplate(
    template='write a details report of {topic} in 100 words',
    input_variables=['topic']
)

In [118]:
# initialize a str output parser
parser = StrOutputParser()

# create two chains with & without parser
chain1 = prompt | model
chain2 = prompt | model | parser

result1 = chain1.invoke({'topic': 'the impact of AI on society'})
result2 = chain2.invoke({'topic': 'the impact of AI on society'})

# compare result
print("Result 1:", result1)
print("Result 2:", result2)

Result 1: content="AI's societal impact is multifaceted. Economically, it fuels automation, potentially displacing jobs while creating new, specialized roles. Socially, AI influences communication through algorithms shaping news and social media feeds, raising concerns about bias and misinformation. Ethically, questions arise regarding privacy, algorithmic fairness, and autonomous decision-making. Furthermore, AI is transforming healthcare with improved diagnostics and personalized treatments. Education, transportation, and entertainment are also undergoing AI-driven transformations, presenting both opportunities and challenges that require careful consideration and proactive policies.\n" response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []} id='run-c2a04673-b972-488e-9d24-855164a55a15-0' usage_metadata={'input_tokens': 17, 'output_tokens': 106, 'total_tokens': 123}
Result 2: AI's impact on society is multifacete

## **Json Output Parser**

In [119]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser

In [120]:
# initialize a json parser
json_parser = JsonOutputParser()

# initialize a prompt template
template = PromptTemplate(
    template = "Give me age, name, city of a fictional person \n {format_instruction}",
    partial_variables = {
        "format_instruction": json_parser.get_format_instructions()
    }
)

prompt = template.format()
print(prompt)

Give me age, name, city of a fictional person 
 Return a JSON object.


In [121]:
# initialize chat model

model = init_chat_model("gemini-2.0-flash", model_provider="google-genai")

In [122]:
response = model.invoke(prompt)
print(response.content)

```json
{
  "name": "Elara Vance",
  "age": 28,
  "city": "Pineshadow, Oregon"
}
```



In [123]:
# parse final Jason object

final_responce = json_parser.parse(response.content)
print(final_responce)

{'name': 'Elara Vance', 'age': 28, 'city': 'Pineshadow, Oregon'}


In [124]:
# create a chain to merge all steps

chain = template | model | json_parser

response = chain.invoke({
    
})
print(response)

{'name': 'Anya Petrova', 'age': 28, 'city': 'St. Petersburg'}


## **Structured Output Parser**

In [125]:
from langchain.output_parsers import StructuredOutputParser, ResponseSchema
from langchain_core.prompts import PromptTemplate

In [126]:
# initialize chat model

model = init_chat_model("gemini-2.0-flash", model_provider="google-genai")

In [127]:
# define schema for output

schema = [
    ResponseSchema(name='fact', description='Fact about the animal'),
    ResponseSchema(name='Age Period', description='Average age period of animal'),
    ResponseSchema(name='Color', description='Color of that animal')
]

# initialize the parser 
parser = StructuredOutputParser.from_response_schemas(schema)

In [128]:
# define a prompt template 

template = PromptTemplate(
    template = "Give details about {topic} \n {format_instruction}" ,
    input_vairables = ["topic"],
    partial_variables = {
        "format_instruction": parser.get_format_instructions()
    }
)

In [129]:
# create a prompt using the template

prompt = template.invoke({
    "topic" : "buffalo"
})

print(prompt.text)

Give details about buffalo 
 The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":

```json
{
	"fact": string  // Fact about the animal
	"Age Period": string  // Average age period of animal
	"Color": string  // Color of that animal
}
```


In [130]:
# get response from model

response = model.invoke(prompt)
print(response.content)

```json
[
  {
    "fact": "Buffalo are herbivores, primarily grazing on grasses and other vegetation.",
    "Age Period": "18-22 years",
    "Color": "Dark brown to black"
  },
  {
    "fact": "There are two main types of buffalo: the water buffalo (Bubalus bubalis) and the African buffalo (Syncerus caffer).",
    "Age Period": "25 years (Water Buffalo in captivity)",
    "Color": "Grayish-black (Water Buffalo)"
  },
  {
    "fact": "African buffalo are known for their aggressive temperament and are considered one of the most dangerous animals in Africa.",
    "Age Period": "11-15 years (African Buffalo in the wild)",
    "Color": "Dark brown or black (African Buffalo)"
  },
  {
    "fact": "Water buffalo are often used as draft animals in agriculture, especially in rice paddies.",
    "Age Period": "25-30 years (Domesticated Water Buffalo)",
    "Color": "Variable, including gray, black, and brown (Domesticated Water Buffalo)"
  },
  {
    "fact": "Buffalo live in herds, which provide

In [136]:
# create a chain to merge all steps

chain = template | model  | parser

response = chain.invoke({
    "topic":"hourse"
})

print(response)

{'fact': 'Horses have a nearly 360-degree panoramic vision, allowing them to see almost everything around them at once. However, they have two blind spots: directly in front of their nose and directly behind them.', 'Age Period': '25-30 years (average)', 'Color': 'Varies widely, including bay, chestnut, black, gray, palomino, roan, and pinto.'}


**Advantage:** It can inforce the structured output in json fromat

**Dis-advantage:** It can't control the data types of output

## **Pydantic Output Parser**

In [137]:
from pydantic import BaseModel, Field
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

In [138]:
# define pydantic class for structured output
class Person(BaseModel):
    Name:str = Field(discription='Name of the person'),
    Age:int = Field(discription='Age of person', gt=18),
    City:str = Field(discription='City name of person blong to')

In [139]:
# define pydantic output parser
parser = PydanticOutputParser(pydantic_object=Person)

In [140]:
# define a template for prompt

template = PromptTemplate(
    template= 'Give details about fictional person of {country} \n {format_instruction}',
    input_variables = ["country"],
    partial_variables = {
        "format_instruction": parser.get_format_instructions()
    }
    
)

/usr/local/lib/python3.11/dist-packages/pydantic/json_schema.py:2324: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=True, json_schema_extra={'discription': 'Name of the person'}),) is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
/usr/local/lib/python3.11/dist-packages/pydantic/json_schema.py:2324: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=True, json_schema_extra={'discription': 'Age of person'}, metadata=[Gt(gt=18)]),) is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)


In [141]:
# create a prompt using template

prompt = template.invoke({
    "country":"africa"
})
print(prompt.text)

Give details about fictional person of africa 
 The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"Name": {"title": "Name", "type": "string"}, "Age": {"title": "Age", "type": "integer"}, "City": {"discription": "City name of person blong to", "title": "City", "type": "string"}}, "required": ["City"]}
```


In [142]:
# get response from model

response = model.invoke(prompt)
final_response = parser.parse(response.content)
print(final_response)

Name='Aisha Diallo' Age=28 City='Dakar'


In [143]:
# create a chain to concate all steps

chain = template | model | parser

chain.invoke({
    'country':"Afganistan"
})

Person(Name='Aria Khan', Age=28, City='Kabul')

 **Advantages:**
 
1. **Strict schema enforcement:** Ensures llm response follow pre-defined structure
2. **Data Type Safty:** Automatically convert llm output into python object

# **Chains**

## **Sequencial Chain**

In [144]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.schema.runnable import RunnableLambda

In [145]:
# initialize model
model = init_chat_model(model='gemini-2.0-flash', model_provider='google-genai')

In [146]:
# define a prompt template
template = PromptTemplate(
    template = 'Tell 2 facts about {topic}',
    input_variables = ["topic"]
)

In [147]:
# define a StrOutputParser
parser = StrOutputParser()

In [148]:
# define a sequential chain

chain = template | model | parser

response = chain.invoke({
    "topic":"buffalo"
})

print(response)

Okay, here are two interesting facts about buffalo (specifically, American Bison):

1.  **They can swim:** Despite their size and bulk, American Bison are strong swimmers and can cross rivers and lakes. This ability was crucial for their survival and migration patterns.

2.  **They create and maintain healthy ecosystems:** Bison grazing habits and wallowing (rolling in dust or mud) create diverse habitats that benefit many other species. Their grazing encourages plant diversity, and wallows provide habitat for insects, amphibians, and birds.



In [149]:
! pip install  grandalf

I0000 00:00:1755363482.028405      36 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 1.2 MB/s eta 0:00:00


In [150]:
# print the structure of chain
chain.get_graph().print_ascii()

      +-------------+      
      | PromptInput |      
      +-------------+      
             *             
             *             
             *             
    +----------------+     
    | PromptTemplate |     
    +----------------+     
             *             
             *             
             *             
+------------------------+ 
| ChatGoogleGenerativeAI | 
+------------------------+ 
             *             
             *             
             *             
    +-----------------+    
    | StrOutputParser |    
    +-----------------+    
             *             
             *             
             *             
+-----------------------+  
| StrOutputParserOutput |  
+-----------------------+  


In [151]:
# define two templates, parser & a model

template1 = PromptTemplate(
    template = "Write 100 words summary about {topic}",
    input_variables = ["topic"]
)

template2 = PromptTemplate(
    template="Give 5 Key points of these details {summary}",
    input_variables = ["summary"]
)

parser = StrOutputParser()

model = init_chat_model(model='gemini-2.0-flash', model_provider="google-genai")

In [152]:
# define a chain

chain = template1 | model | parser | template2 | model | parser

In [153]:
response = chain.invoke({
    "topic":"Indian Education System"
})

print(response)

Here are 5 key points summarizing the provided details about the Indian education system:

1.  **Dualistic System:** India's education system blends ancient traditions with modern, global influences, operating through a mix of public and private institutions.
2.  **Access Disparities:** Despite the goal of universal access, significant inequalities persist based on location (urban vs. rural) and socioeconomic status.
3.  **Standardized Structure:** The system is structured with primary, secondary, and higher education levels, marked by standardized curricula and high-stakes examinations (e.g., 10th and 12th board exams).
4.  **Persistent Challenges:** Despite progress in enrollment, the system faces challenges like teacher shortages, poor infrastructure, and an over-reliance on rote learning.
5.  **Reform Focus:** Recent reforms prioritize skill development and vocational training to enhance employability and address the shortcomings of traditional education.



In [154]:
# chain structure
chain.get_graph().print_ascii()

      +-------------+      
      | PromptInput |      
      +-------------+      
             *             
             *             
             *             
    +----------------+     
    | PromptTemplate |     
    +----------------+     
             *             
             *             
             *             
+------------------------+ 
| ChatGoogleGenerativeAI | 
+------------------------+ 
             *             
             *             
             *             
    +-----------------+    
    | StrOutputParser |    
    +-----------------+    
             *             
             *             
             *             
+-----------------------+  
| StrOutputParserOutput |  
+-----------------------+  
             *             
             *             
             *             
    +----------------+     
    | PromptTemplate |     
    +----------------+     
             *             
             *             
             *      

## **Parrallel Chain**

In [155]:
# initialize chat model

model = init_chat_model(model='gemini-2.0-flash', model_provider='google-genai')

In [156]:
# define templates

template1 = PromptTemplate(
    template = "Generate simple & consice notes from this text {text}",
    input_variables = ["text"]
)

template2 = PromptTemplate(
    template = "Generate 5 Consice Q&As from this text {text}",
    intput_variables=["text"]
)

template3 = PromptTemplate(
    template = "Merge the provided notes & quiz into single document \n notes : {notes} , quiz: {quiz}",
    input_variables = ['notes', 'quiz']
)

In [157]:
# define a output parser
parser = StrOutputParser()

In [158]:
# define chain
from langchain.schema.runnable import RunnableParallel

# define a parallel chain
parallel_chain = RunnableParallel({
    'notes': template1 | model | parser,
    'quiz': template2 | model | parser
})

# define chain at merge chain
merge_chain = template3 | model | parser

# define a combined chain
chain = parallel_chain | merge_chain

In [159]:
response = chain.invoke({
    "text":"machine learning"
})

print(response)

# Machine Learning: Notes and Quiz

## Machine Learning: Concise Notes

**What it is:**

*   Algorithms that learn from data without explicit programming.
*   Goal: Predict or make decisions based on data.

**Key Types:**

*   **Supervised Learning:** Learns from labeled data (input & correct output).
    *   **Regression:** Predicts continuous values (e.g., price).
    *   **Classification:** Predicts categories (e.g., spam/not spam).
*   **Unsupervised Learning:** Learns from unlabeled data (only input).
    *   **Clustering:** Groups similar data points (e.g., customer segmentation).
    *   **Dimensionality Reduction:** Reduces the number of variables.
*   **Reinforcement Learning:** Learns through trial and error, receiving rewards/penalties.

**General Process:**

1.  **Data Collection & Preparation:** Gather and clean data.
2.  **Model Selection:** Choose appropriate algorithm.
3.  **Training:** Train the model using data.
4.  **Evaluation:** Assess model performance.
5.  **Depl

In [160]:
# print chain structure
chain.get_graph().print_ascii()

                    +---------------------------+                      
                    | Parallel<notes,quiz>Input |                      
                    +---------------------------+                      
                       ***                   ***                       
                   ****                         ****                   
                 **                                 **                 
    +----------------+                          +----------------+     
    | PromptTemplate |                          | PromptTemplate |     
    +----------------+                          +----------------+     
             *                                           *             
             *                                           *             
             *                                           *             
+------------------------+                  +------------------------+ 
| ChatGoogleGenerativeAI |                  | ChatGoogleGenerati

## **Conditional Chain**

In [161]:
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field
from typing import Literal
from langchain.schema.runnable import RunnableBranch, RunnableLambda

In [162]:
# initialize chat model

model = init_chat_model(model='gemini-2.0-flash', model_provider='google-genai')

In [163]:
#define structure
class Sentiment(BaseModel):
    sentiment: Literal['Positive', 'Negative'] = Field(description='Give the sentiment of this Feedback')

# define parser
parser1 = StrOutputParser()

parser2 = PydanticOutputParser(pydantic_object=Sentiment)

# define templates
template1 = PromptTemplate(
    template="Classify the senitment of following feedback into positive or negative \n {feedback} \n {format_instruction}",
    input_variables=["feedback"],
    partial_variables = {
        "format_instruction" : parser2.get_format_instructions()
    }
)

template2 = PromptTemplate(
    template="Behave like a customer service officer, give appropriate response for customer of postive response of this {feedback}", 
    input_variables = ["feedback"]
)

template3 = PromptTemplate(
    template="Behave like a customer service officer, give appropriate response for custormer of negative response of this {feedback}", 
    input_variables = ["feedback"]
)

In [164]:
# Modified classifier_chain to output both sentiment and feedback
classifier_chain = (
    {
        "feedback": lambda x: x["feedback"],  # keep original feedback
        "sentiment": template1 | model | parser2
    }
)

# Now branch_chain can use both
branch_chain = RunnableBranch(
    (lambda x: x["sentiment"].sentiment == 'Positive',
        {"feedback": lambda x: x["feedback"]} | template2 | model | parser1
    ),
    (lambda x: x["sentiment"].sentiment == 'Negative',
        {"feedback": lambda x: x["feedback"]} | template3 | model | parser1
    ),
    RunnableLambda(lambda x: "Could not find sentiment")
)

In [165]:
# invoke defined conditional chain
chain = classifier_chain | branch_chain

# Example run
result = chain.invoke({"feedback": "I love this product, it works perfectly!"})
print(result)

Okay, great! Here are a few options for responding, depending on how much you want to engage with the customer:

**Option 1 (Simple and Efficient):**

> "That's fantastic to hear! We're so glad you're loving the product and that it's working perfectly for you. Thanks for letting us know!"

**Option 2 (Adding a Personal Touch):**

> "Wonderful! It makes our day to hear that you're happy with the product. We put a lot of effort into making sure it works well. Is there anything else we can help you with today?"

**Option 3 (Encouraging Future Engagement):**

> "Awesome! We're thrilled you're having such a positive experience. We'd love it if you considered leaving a review on our website or social media! It really helps others discover our products. Thanks again!"

**Option 4 (Highlighting Other Products, if appropriate):**

> "That's amazing! We're so happy to hear that it's working perfectly for you. If you're interested in other products that might complement [the product they're talki

# **Runnables**

## **Runnable Sequence**

In [166]:
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain.schema.runnable import RunnableSequence

In [167]:
# initialize a chat model

model = init_chat_model(model='gemini-2.0-flash', model_provider = 'google-genai')

In [168]:
# initialize parser & template

template = PromptTemplate(
    template = "Write a Joke on {topic}", 
    input_variables = ['topic']
)

parser = StrOutputParser()

In [169]:
# create chain using Runnable Sequencial

chain = RunnableSequence(template, model, parser)

response = chain.invoke({
    "topic":"Buffalo"
})

print(response)

Why did the buffalo cross the road?

To get to the Buffalo Wild Wings on the other side!



## **Runnable Parallel**

In [170]:
from langchain.schema.runnable import RunnableParallel

In [171]:
# initialize chat model

model = init_chat_model(model='gemini-2.0-flash',model_provider='google-genai' )

In [172]:
# initialize templates

template1 = PromptTemplate(
    template = "Generate a linked-in post about {topic}",
    input_variables=['topic']
)

template2 = PromptTemplate(
    template = "Generate a X post about {topic}",
    input_variables=['topic']
)

# initialize a parser

parser = StrOutputParser()

In [173]:
# initialize parallel chain

parallel_chain = RunnableParallel({
    "linked-in":template1 | model | parser,
    "X": template2 | model | parser
})

In [174]:
response = parallel_chain.invoke({
    "topic":"impact on economy of AI"
})

print(response['X'])
print(response['linked-in'])

AI is poised to reshape the economy! 🤖 From boosting productivity & creating new jobs to potentially displacing others, the impact will be massive. We need to focus on reskilling, ethical development, & equitable access to ensure AI benefits everyone. #AI #Economy #FutureofWork #Innovation

## AI: A Double-Edged Sword with Massive Economic Impact ⚔️💰

Artificial Intelligence is no longer a futuristic fantasy; it's reshaping our economy in profound ways. From automating mundane tasks to driving innovation in complex industries, AI's influence is undeniable. But is it all sunshine and roses?

**Here's a quick look at the economic impact of AI:**

**The Good:**

*   **Increased Productivity:** AI-powered automation streamlines processes, boosting productivity and efficiency across various sectors.
*   **New Job Creation:** While some jobs may be displaced, AI is also creating new roles in areas like AI development, data science, and AI ethics.
*   **Economic Growth:** By driving innovatio

## **Runnable Passthrough**

In [175]:
from langchain.schema.runnable import RunnablePassthrough, RunnableSequence, RunnableParallel

In [176]:
# initialize chat model

model = init_chat_model(model='gemini-2.0-flash',model_provider='google-genai' )

In [177]:
# initialize templates

template1 = PromptTemplate(
    template = "Generate a joke on {topic}",
    input_variables=['topic']
)

template2 = PromptTemplate(
    template = "Explain this joke,  {joke}",
    input_variables=['topic']
)

# initialize a parser

parser = StrOutputParser()

In [178]:
# define chains

chain1 = RunnableSequence(template1, model, parser)

chain2 = RunnableParallel({
    "Joke": RunnablePassthrough(),
    "explanation" : RunnableSequence(template2, model, parser)
})

final_chain = chain1 | chain2

In [179]:
response = final_chain.invoke({"topic":"cricket"})
print(response)

{'Joke': 'Why did the cricket team bring a ladder to the game?\n\nBecause they heard the batting order was going to be sky high!\n', 'explanation': 'The joke plays on the double meaning of "sky high."\n\n*   **Literal Meaning:** "Sky high" refers to something being very tall or reaching a great height. This is why the cricket team brought a ladder - they literally thought they\'d need to climb to reach the batting order.\n\n*   **Figurative Meaning:** "Sky high" can also mean something is very expensive or of a very high standard. In this case, the joke implies the batting order (the list of players and their position in the batting lineup) is going to be exceptionally good or impressive.\n\nThe humor comes from the misunderstanding of the phrase "sky high" and the absurd image of a cricket team bringing a ladder to a game.\n'}


## **Runnable Lambda**

In [180]:
from langchain.schema.runnable import RunnableLambda, RunnablePassthrough, RunnableSequence, RunnableParallel

In [181]:
# define a function

def counter(sentence):
    return len(sentence.split(' '))

In [182]:
# initialize chat model

model = init_chat_model(model='gemini-2.0-flash',model_provider='google-genai' )

In [183]:
# initialize templates

template1 = PromptTemplate(
    template = "Generate a joke on {topic}",
    input_variables=['topic']
)

template2 = PromptTemplate(
    template = "Explain this joke,  {joke}",
    input_variables=['topic']
)

# initialize a parser

parser = StrOutputParser()

In [184]:
# define chains

chain1 = RunnableSequence(template1, model, parser)

chain2 = RunnableParallel({
    "Joke": RunnablePassthrough(),
    "Number of words" : RunnableLambda(counter)
})

final_chain = chain1 | chain2

In [185]:
response = final_chain.invoke({"topic":"cricket"})
print(response)

{'Joke': 'Why did the cricket team bring a ladder to the game?\n\nBecause they heard the runs were on the board! \n', 'Number of words': 20}


## **Runnable Branch**

In [186]:
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field
from typing import Literal
from langchain.schema.runnable import RunnableBranch, RunnableLambda

In [187]:
# initialize chat model
model = init_chat_model(model='gemini-2.0-flash', model_provider='google-genai')

In [188]:
#define structure
class Sentiment(BaseModel):
    sentiment: Literal['Positive', 'Negative'] = Field(description='Give the sentiment of this Feedback')

# define parser
parser1 = StrOutputParser()

parser2 = PydanticOutputParser(pydantic_object=Sentiment)

# define templates
template1 = PromptTemplate(
    template="Classify the senitment of following feedback into positive or negative \n {feedback} \n {format_instruction}",
    input_variables=["feedback"],
    partial_variables = {
        "format_instruction" : parser2.get_format_instructions()
    }
)

template2 = PromptTemplate(
    template="Behave like a customer service officer & give 1 lines appropriate response for customer of postive response of this {feedback}", 
    input_variables = ["feedback"]
)

template3 = PromptTemplate(
    template="Behave like a customer service officer, give 1 lines appropriate response for custormer of negative response of this {feedback}", 
    input_variables = ["feedback"]
)

In [189]:
# Modified classifier_chain to output both sentiment and feedback
classifier_chain = (
    {
        "feedback": lambda x: x["feedback"],  # keep original feedback
        "sentiment": template1 | model | parser2
    }
)

# Now branch_chain can use both
branch_chain = RunnableBranch(
    (lambda x: x["sentiment"].sentiment == 'Positive',
        {"feedback": lambda x: x["feedback"]} | template2 | model | parser1
    ),
    (lambda x: x["sentiment"].sentiment == 'Negative',
        {"feedback": lambda x: x["feedback"]} | template3 | model | parser1
    ),
    RunnableLambda(lambda x: "Could not find sentiment")
)

In [190]:
# invoke defined conditional chain
chain = classifier_chain | branch_chain

# Example run
result = chain.invoke({"feedback": "I love this product, it works perfectly!"})
print(result)

"We're so happy to hear you're loving it and that it's working perfectly for you!"



# **Document Loders**

## **Text Loader**

In [191]:
from langchain_community.document_loaders import TextLoader

In [192]:
# initialize loader
loader = TextLoader('/kaggle/input/poem-file/poem.txt', encoding='utf-8')

In [ ]:
# load data 
doc = loader.load()

In [ ]:
# data of docs 

for page in doc:
    print(page.page_content)
    print(page.metadata)

In [ ]:
# summarize this page content using llm

model = init_chat_model(model='gemini-2.0-flash', model_provider='google-genai')

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

In [ ]:
template = PromptTemplate(
    template = "Summarize this poem in two lines {poem}",
    input_variables = ["poem"]
)

parser = StrOutputParser()

In [ ]:
chain = template | model | parser

response = chain.invoke({
    "poem":doc[0].page_content
})

print(response)

## **PyPDF Loader**

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

In [ ]:
# initialize loader
loader = PyPDFLoader('/kaggle/input/poem-file/Harsh_Resume_IITP.pdf')

In [ ]:
#  load document 
doc = loader.load()

In [ ]:
for page in doc:
    print(page.metadata)
    print(page.page_content)

In [ ]:
# create a small template to review any resume

template = PromptTemplate(
    template="Review this resume care fully & provide painpoints, suggestion for improvement and rate resume out of 10. here is the resume content {resume}",
    input_variables = ["resume"]
)

model = init_chat_model(model='gemini-2.0-flash', model_provider='google-genai')

parser = StrOutputParser()

In [ ]:
chain = template | model | parser

response = chain.invoke({
    "resume":doc[0].page_content
})

print(response)

## **Directory Loader**

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader

In [ ]:
# initialize loader 
loader = DirectoryLoader(
    path='/kaggle/input/poem-file',
    glob='.pdf',
    loader_cls=PyPDFLoader
)

doc = loader.load()

## **Web Base Loader**

In [195]:
from langchain_community.document_loaders import WebBaseLoader

In [196]:
# initialize loader
loader = WebBaseLoader(
    ['https://gate2026.iitg.ac.in/index.html', 'https://gate2026.iitg.ac.in/important-dates.html']
)

In [197]:
doc = loader.load()

In [198]:
doc[0].page_content

"\n\n\n\n\n\n\nGATE 2026\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nGRADUATE APTITUDE TEST IN ENGINEERING 2026\nअभियांत्रिकी स्नातक अभिक्षमता परीक्षा 2026\nOrganizing Institute : INDIAN INSTITUTE OF TECHNOLOGY GUWAHATI\n\n\n\n\n\n\n\n\n\n\n\nHome\nImportant Dates\nGATE PAPERS \n\nGATE 2026 Test PAPERS & Syllabus\nQuestion Paper Pattern\nGATE 2026 Two-Paper Combinations\nGATE 2026 Examination Schedule\n\n\n\nExamination\n\nOpportunities\nEligibility Criteria\nApplication Fees\nExam Cities\n\nPhotograph & Signature\nRequired Documents\n\n\n\nDownload\nFAQS\nContact\n\nAPPLICATION PORTAL \n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nNOTIFICATIONS \n\n\n\n\n\n              New sectional paper on Energy Science (XE-I) is introduced in Engineering Sciences (XE) paper.\n          \n\n\n\n\n\n\n\n\nGATE 2026\n\n\nGraduate Aptitude Test in Engineering (GATE) is a prestigious national-level examination.\nThe examination assesses the candidates for a comprehensive unders

## **CSV Loader**

In [199]:
from langchain_community.document_loaders import CSVLoader

In [ ]:
# initialize loader
loader = CSVLoader(
    file_path = '/kaggle/input/poem-file/submission (2).csv'
)

In [ ]:
# load document
doc = loader.load()

In [ ]:
print(doc[0])

In [ ]:
%config Completer.use_jedi = False

# **Text Splitters**

## **Character Text Splitter**

In [200]:
from langchain.text_splitter import CharacterTextSplitter

In [201]:
text = "\n\n\n\n\n\n\nGATE 2026\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nGRADUATE APTITUDE TEST IN ENGINEERING 2026\nअभियांत्रिकी स्नातक अभिक्षमता परीक्षा 2026\nOrganizing Institute : INDIAN INSTITUTE OF TECHNOLOGY GUWAHATI\n\n\n\n\n\n\n\n\n\n\n\nHome\nImportant Dates\nGATE PAPERS \n\nGATE 2026 Test PAPERS & Syllabus\nQuestion Paper Pattern\nGATE 2026 Two-Paper Combinations\nGATE 2026 Examination Schedule\n\n\n\nExamination\n\nOpportunities\nEligibility Criteria\nApplication Fees\nExam Cities\n\nPhotograph & Signature\nRequired Documents\n\n\n\nDownload\nFAQS\nContact\n\nAPPLICATION PORTAL \n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nNOTIFICATIONS \n\n\n\n\n\n              New sectional paper on Energy Science (XE-I) is introduced in Engineering Sciences (XE) paper.\n          \n\n\n\n\n\n\n\n\nGATE 2026\n\n\nGraduate Aptitude Test in Engineering (GATE) is a prestigious national-level examination.\nThe examination assesses the candidates for a comprehensive understanding of different undergraduate-level subjects in Engineering/Technology/Science/Commerce/Arts/Architecture/Humanities.\nGATE 2026 is being conducted by IISc and all IITs, on behalf of the National Coordination Board (NCB), Department of Higher Education, Ministry of Education (MoE), Government of India.\nIIT Guwahati is the Organizing Institute for GATE 2026.\n\n\n\n\n\n\n\nOPPORTUNITIES\n\nCandidates who qualify for GATE can seek admission with possible financial assistance to Master's programs, Direct Doctoral programs and Doctoral programs in relevant branches of Engineering/Technology/Science/Architecture/Humanities in institutions supported by Ministry of Education (MoE) and other Government agencies.\nGATE score is also used by some colleges and Institutions for admission to postgraduate programs without MoE scholarship.\nSeveral Public Sector Undertakings (PSUs) have also been using GATE score for recruitment.\n\n\n\n\n\nGATE 2026 TEST PAPERS\n\nGATE 2026 will have a total of 30 test papers comprising full and sectional papers.\nA new sectional paper on Energy Science (XE-I) is introduced in Engineering Sciences (XE) paper.\nA candidate can appear for one or two test papers.\nOnly selected two-paper combinations are allowed for candidates appearing in two papers.\nGATE score obtained by the candidate will remain valid for a period of THREE years from the date of announcement of results.\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nIMPORTANT DATES\n\n  August 25, 2025 :  ONLINE REGISTRATION OPENS\n  September 25, 2025 :  ONLINE REGISTRATION CLOSES (WITHOUT LATE FEE)\n  October 6, 2025 :  ONLINE REGISTRATION CLOSES (WITH LATE FEE)\n  March 19, 2026 :  ANNOUNCEMENT OF RESULTS\n Dates are liable to change.\n\nView All Dates\n\n\n\n\n\n\n\n\n\n\n\n\nExamination Schedule\nGATE 2026 Examination Schedule\n\n\n\n\n\n\n\nExamination Cities\nThere are no international centres for GATE 2026 examination.\n\n\n\n\n\n\n\nEligibility Criteria\nBefore filing an application, candidates are advised to ensure that they meet the eligibility criteria for GATE 2026\n\n\n\n\n\n\n\n\nApplication Portal\n\nClick here to go to the Application portal for GATE 2026\n\n\n\n\n\n\n\nRequired Documents\nBefore filing an application, candidates are advised to check Data Required for Filling the Online Application Form in GATE 2026\n\n\n\n\n\n\n\nTwo-Paper Combinations\n\nTwo-paper combinations will be updated soon.\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nGATE 2026\n\n\nGATE-JAM Office\nIIT Guwahati\nPhone: +91 361 258 6500\nEmail: helpdesk.gate@iitg.ac.in\n\n\n\n\n\n\n\n\n\n\nIMPORTANT LINKS\n\nImportant Dates\n Application Portal\nExamination Cities\nEligibility Criteria\nApplication Fees\n\n\n\nQR Code\n\n\n\n\n\n\nDISCLAIMER\n\n In case of any unforeseen circumstances beyond control, the exam may be postponed or cancelled.\n Qualifying in GATE does not guarantee admission, scholarship, or a job. Admission to any institute is fully dependent on the admitting institute’s criteria for educational qualification. GATE qualification does not assure a public sector undertaking (PSU) job. No responsibility is assumed for admission, scholarship, or a job.\n\n\n\n\n\n© GATE 2026, Indian Institute of Technology Guwahati. All Rights Reserved.\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n"

splitter = CharacterTextSplitter(
    chunk_size=100, 
    chunk_overlap=0,
    separator = ''
)

result = splitter.split_text(text)

In [202]:
print(result[6])

onal paper on Energy Science (XE-I) is introduced in Engineering Sciences (XE) paper.


## **Recursive Character Text Splitter**

In [203]:
from langchain.text_splitter import RecursiveCharacterTextSplitter, Language

In [204]:
text = "\n\n\n\n\n\n\nGATE 2026\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nGRADUATE APTITUDE TEST IN ENGINEERING 2026\nअभियांत्रिकी स्नातक अभिक्षमता परीक्षा 2026\nOrganizing Institute : INDIAN INSTITUTE OF TECHNOLOGY GUWAHATI\n\n\n\n\n\n\n\n\n\n\n\nHome\nImportant Dates\nGATE PAPERS \n\nGATE 2026 Test PAPERS & Syllabus\nQuestion Paper Pattern\nGATE 2026 Two-Paper Combinations\nGATE 2026 Examination Schedule\n\n\n\nExamination\n\nOpportunities\nEligibility Criteria\nApplication Fees\nExam Cities\n\nPhotograph & Signature\nRequired Documents\n\n\n\nDownload\nFAQS\nContact\n\nAPPLICATION PORTAL \n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nNOTIFICATIONS \n\n\n\n\n\n              New sectional paper on Energy Science (XE-I) is introduced in Engineering Sciences (XE) paper.\n          \n\n\n\n\n\n\n\n\nGATE 2026\n\n\nGraduate Aptitude Test in Engineering (GATE) is a prestigious national-level examination.\nThe examination assesses the candidates for a comprehensive understanding of different undergraduate-level subjects in Engineering/Technology/Science/Commerce/Arts/Architecture/Humanities.\nGATE 2026 is being conducted by IISc and all IITs, on behalf of the National Coordination Board (NCB), Department of Higher Education, Ministry of Education (MoE), Government of India.\nIIT Guwahati is the Organizing Institute for GATE 2026.\n\n\n\n\n\n\n\nOPPORTUNITIES\n\nCandidates who qualify for GATE can seek admission with possible financial assistance to Master's programs, Direct Doctoral programs and Doctoral programs in relevant branches of Engineering/Technology/Science/Architecture/Humanities in institutions supported by Ministry of Education (MoE) and other Government agencies.\nGATE score is also used by some colleges and Institutions for admission to postgraduate programs without MoE scholarship.\nSeveral Public Sector Undertakings (PSUs) have also been using GATE score for recruitment.\n\n\n\n\n\nGATE 2026 TEST PAPERS\n\nGATE 2026 will have a total of 30 test papers comprising full and sectional papers.\nA new sectional paper on Energy Science (XE-I) is introduced in Engineering Sciences (XE) paper.\nA candidate can appear for one or two test papers.\nOnly selected two-paper combinations are allowed for candidates appearing in two papers.\nGATE score obtained by the candidate will remain valid for a period of THREE years from the date of announcement of results.\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nIMPORTANT DATES\n\n  August 25, 2025 :  ONLINE REGISTRATION OPENS\n  September 25, 2025 :  ONLINE REGISTRATION CLOSES (WITHOUT LATE FEE)\n  October 6, 2025 :  ONLINE REGISTRATION CLOSES (WITH LATE FEE)\n  March 19, 2026 :  ANNOUNCEMENT OF RESULTS\n Dates are liable to change.\n\nView All Dates\n\n\n\n\n\n\n\n\n\n\n\n\nExamination Schedule\nGATE 2026 Examination Schedule\n\n\n\n\n\n\n\nExamination Cities\nThere are no international centres for GATE 2026 examination.\n\n\n\n\n\n\n\nEligibility Criteria\nBefore filing an application, candidates are advised to ensure that they meet the eligibility criteria for GATE 2026\n\n\n\n\n\n\n\n\nApplication Portal\n\nClick here to go to the Application portal for GATE 2026\n\n\n\n\n\n\n\nRequired Documents\nBefore filing an application, candidates are advised to check Data Required for Filling the Online Application Form in GATE 2026\n\n\n\n\n\n\n\nTwo-Paper Combinations\n\nTwo-paper combinations will be updated soon.\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nGATE 2026\n\n\nGATE-JAM Office\nIIT Guwahati\nPhone: +91 361 258 6500\nEmail: helpdesk.gate@iitg.ac.in\n\n\n\n\n\n\n\n\n\n\nIMPORTANT LINKS\n\nImportant Dates\n Application Portal\nExamination Cities\nEligibility Criteria\nApplication Fees\n\n\n\nQR Code\n\n\n\n\n\n\nDISCLAIMER\n\n In case of any unforeseen circumstances beyond control, the exam may be postponed or cancelled.\n Qualifying in GATE does not guarantee admission, scholarship, or a job. Admission to any institute is fully dependent on the admitting institute’s criteria for educational qualification. GATE qualification does not assure a public sector undertaking (PSU) job. No responsibility is assumed for admission, scholarship, or a job.\n\n\n\n\n\n© GATE 2026, Indian Institute of Technology Guwahati. All Rights Reserved.\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n"

splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=5
)

In [205]:
result = splitter.split_text(text)
print(len(result))

17


In [206]:
result[15]

'Qualifying in GATE does not guarantee admission, scholarship, or a job. Admission to any institute is fully dependent on the admitting institute’s criteria for educational qualification. GATE qualification does not assure a public sector undertaking (PSU) job. No responsibility is assumed for admission, scholarship, or a job.'

In [207]:
code = """
class Dog:
    # Class attribute
    species = "Canis familiaris"

    # Constructor method (initializer)
    def __init__(self, name, breed):
        self.name = name  # Instance attribute
        self.breed = breed  # Instance attribute

    # Instance method
    def bark(self):
        return f"{self.name} says Woof!"

    # Another instance method
    def describe(self):
        return f"{self.name} is a {self.breed} {self.species}."

# Creating objects (instances) of the Dog class
dog1 = Dog("Buddy", "Golden Retriever")
dog2 = Dog("Max", "German Shepherd")

# Accessing instance attributes
print(f"Dog 1's name: {dog1.name}")
print(f"Dog 2's breed: {dog2.breed}")

# Calling instance methods
print(dog1.bark())
print(dog2.describe())

# Accessing class attribute
print(f"All dogs belong to the species: {Dog.species}")
"""

In [208]:
splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON,
    chunk_size=300,
    chunk_overlap=0
    
)

In [209]:
result = splitter.split_text(code)
print(len(result))

4


In [210]:
print(result[0])

class Dog:
    # Class attribute
    species = "Canis familiaris"

    # Constructor method (initializer)
    def __init__(self, name, breed):
        self.name = name  # Instance attribute
        self.breed = breed  # Instance attribute


## **Semantic Meaning Based Text Splitter**

In [211]:
text = """
Artificial intelligence is transforming the way we work, learn, and communicate. From chatbots to 
autonomous vehicles, it’s making tasks faster and more efficient. However, it also raises important 
questions about ethics and job displacement.
The rainforest is often called the “lungs of the Earth” because it produces vast amounts of oxygen.
Its diverse ecosystem is home to countless species found nowhere else. Protecting it is essential 
for global environmental balance.
Every small step you take towards your goal counts. Consistency beats intensity when it comes to 
achieving long-term success. Believe in progress, not perfection."""

In [212]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [213]:
# Initialize the Google Generative AI embeddings model
google_embeddings = GoogleGenerativeAIEmbeddings(
    model='models/gemini-embedding-exp-03-07',  # Specify the model name
)

In [214]:
text_splitter = SemanticChunker(
    google_embeddings, 
    breakpoint_threshold_type='standard_deviation',
    breakpoint_threshold_amount=0.8
)

In [215]:
result = text_splitter.split_text(text)

In [216]:
print(len(result))

3


In [217]:
print(result)

['\nArtificial intelligence is transforming the way we work, learn, and communicate. From chatbots to \nautonomous vehicles, it’s making tasks faster and more efficient.', 'However, it also raises important \nquestions about ethics and job displacement. The rainforest is often called the “lungs of the Earth” because it produces vast amounts of oxygen. Its diverse ecosystem is home to countless species found nowhere else.', 'Protecting it is essential \nfor global environmental balance. Every small step you take towards your goal counts. Consistency beats intensity when it comes to \nachieving long-term success. Believe in progress, not perfection.']


# **Vector Stores**

In [218]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain.schema import Document

In [219]:
# create langchain documents

doc1 = Document(
    page_content = '''Mumbai Indians are the most successful IPL team with five championship titles.
Known for their fierce comebacks, they blend young talent with seasoned stars.
Rohit Sharma has been their iconic leader, shaping the team's winning mentality.
They have a massive fan base and a reputation for dominating under pressure.''',
    metadata = {"team":'Mumbai Indians'}
)

doc2 =  Document(
    page_content = '''CSK, led by the legendary MS Dhoni, are synonymous with consistency and resilience.
They have lifted the IPL trophy multiple times, earning respect across the league.
The team is famous for its “Dad’s Army” tag, with experienced players delivering big.
Their “Whistle Podu” fans create one of the most passionate atmospheres in cricket.''',
    metadata = {"team":'Chennai Super Kings'}
)

doc3 =   Document(
    page_content = '''RCB are known for star power, with legends like Virat Kohli, AB de Villiers, and Chris Gayle.
They’re famous for thrilling chases and explosive batting line-ups.
Despite never winning an IPL trophy, they have a loyal global fan following.
Their motto “Ee Sala Cup Namde” reflects their relentless pursuit of glory.''',
    metadata = {"team":'Royal Challengers Bengaluru'}
)

doc4 =   Document(
    page_content = '''Owned by Bollywood superstar Shah Rukh Khan, KKR blends entertainment with cricket.
They’ve won two IPL titles under the leadership of Gautam Gambhir.
Known for their purple-and-gold theme, they play with aggressive intent.
Their fan anthem “Korbo Lorbo Jeetbo” inspires both players and supporters.''',
    metadata = {"team":'Kolkata Knight Riders'}
)

doc5 =   Document(
    page_content = '''Rajasthan Royals were the inaugural IPL champions in 2008 under Shane Warne.
They focus on unearthing young talent and giving them big opportunities.
RR is known for their bold team strategies and unpredictable match performances.
With stars like Jos Buttler and Sanju Samson, they remain a dangerous side.''',
    metadata = {"team":'Rajasthan Royals'}
)

In [220]:
docs = [doc1, doc2, doc3, doc4, doc5]

In [221]:
from langchain_community.vectorstores import Chroma

vector_store = Chroma(
    embedding_function=google_embeddings,
    persist_directory='chroma_db',
    collection_name='IPLTeamDetails'
)


In [222]:
vector_store.add_documents(docs)

['f684e16f-73bb-4a57-927a-861ca3800856',
 '1185ca19-7eca-430f-9215-cb28cc1e2793',
 'd7b41407-0812-49b0-bc71-219bc14f268f',
 '7858e779-aa45-4699-8929-845d4d2cfac6',
 '2038dc47-ff1f-416d-ad97-e8a6d290be82']

In [223]:
vector_store.get(include=['embeddings', 'documents', 'metadatas'])

{'ids': ['03ab2e99-0d06-43fe-a68d-eea0c7c082a6',
  'f68940e4-ec44-4f82-9fa2-7c5db3fc31e8',
  'd40de81c-10be-436a-b8b9-e538a00458f4',
  'f989c534-0be3-485d-824e-bfd77fb6aa55',
  '723ff32a-2116-49dc-b08a-bc47f0b25d9c',
  '2159828c-a640-4c3d-a596-ae1ab17ac7ba',
  '6fcea4ad-034f-486a-9444-dfb70a78504b',
  '525dcfa8-272d-48d2-b944-ec27d71b4676',
  'f684e16f-73bb-4a57-927a-861ca3800856',
  '1185ca19-7eca-430f-9215-cb28cc1e2793',
  'd7b41407-0812-49b0-bc71-219bc14f268f',
  '7858e779-aa45-4699-8929-845d4d2cfac6',
  '2038dc47-ff1f-416d-ad97-e8a6d290be82'],
 'embeddings': array([[-0.00557552, -0.0078158 ,  0.00561287, ...,  0.0160321 ,
         -0.00545004,  0.01238281],
        [-0.00708478, -0.0009398 ,  0.02365722, ...,  0.01071198,
          0.00236517,  0.00041986],
        [ 0.01347261,  0.00961961,  0.01786358, ...,  0.00545023,
          0.00166906,  0.00190981],
        ...,
        [-0.00708478, -0.0009398 ,  0.02365722, ...,  0.01071198,
          0.00236517,  0.00041986],
        [ 0

In [224]:
# similarity search 

vector_store.similarity_search(
    query='who amoung these are bowler?',
    k=1
)

[Document(metadata={'team': 'Rajasthan Royals'}, page_content='Rajasthan Royals were the inaugural IPL champions in 2008 under Shane Warne.\nThey focus on unearthing young talent and giving them big opportunities.\nRR is known for their bold team strategies and unpredictable match performances.\nWith stars like Jos Buttler and Sanju Samson, they remain a dangerous side.')]

In [225]:
# similarity search with score

vector_store.similarity_search_with_score(
    query='who amoung these are bowler?',
    k=1,
    filter={'team':'Rajasthan Royals'} # filter to seach on specific teams
)


[(Document(metadata={'team': 'Rajasthan Royals'}, page_content='Rajasthan Royals were the inaugural IPL champions in 2008 under Shane Warne.\nThey focus on unearthing young talent and giving them big opportunities.\nRR is known for their bold team strategies and unpredictable match performances.\nWith stars like Jos Buttler and Sanju Samson, they remain a dangerous side.'),
  0.7215002179145813)]

In [226]:
# update document

new_doc = Document(
    page_content='''Sunrisers Hyderabad are known for their solid bowling attack and disciplined game plan.
They won the IPL title in 2016 under the leadership of David Warner.
The team often focuses on defending modest totals with exceptional precision.
Their “Orange Army” supporters bring vibrant energy to every match.''',
    metadata={'team':'Sunrisers Hyderabad'}
)

vector_store.update_document(document_id='49924280-8d66-4d51-9669-f361d1a38c07', document=new_doc)

In [227]:
# delete documents

vector_store.delete(ids=['49924280-8d66-4d51-9669-f361d1a38c07'])

In [228]:
vector_store.get(include=['embeddings'])

{'ids': ['03ab2e99-0d06-43fe-a68d-eea0c7c082a6',
  'f68940e4-ec44-4f82-9fa2-7c5db3fc31e8',
  'd40de81c-10be-436a-b8b9-e538a00458f4',
  'f989c534-0be3-485d-824e-bfd77fb6aa55',
  '723ff32a-2116-49dc-b08a-bc47f0b25d9c',
  '2159828c-a640-4c3d-a596-ae1ab17ac7ba',
  '6fcea4ad-034f-486a-9444-dfb70a78504b',
  '525dcfa8-272d-48d2-b944-ec27d71b4676',
  'f684e16f-73bb-4a57-927a-861ca3800856',
  '1185ca19-7eca-430f-9215-cb28cc1e2793',
  'd7b41407-0812-49b0-bc71-219bc14f268f',
  '7858e779-aa45-4699-8929-845d4d2cfac6',
  '2038dc47-ff1f-416d-ad97-e8a6d290be82'],
 'embeddings': array([[-0.00557552, -0.0078158 ,  0.00561287, ...,  0.0160321 ,
         -0.00545004,  0.01238281],
        [-0.00708478, -0.0009398 ,  0.02365722, ...,  0.01071198,
          0.00236517,  0.00041986],
        [ 0.01347261,  0.00961961,  0.01786358, ...,  0.00545023,
          0.00166906,  0.00190981],
        ...,
        [-0.00708478, -0.0009398 ,  0.02365722, ...,  0.01071198,
          0.00236517,  0.00041986],
        [ 0

# **Retrievers**

## **Wikipedia Retriever**

In [229]:
!pip install wikipedia

I0000 00:00:1755363622.213441      36 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers


  Preparing metadata (setup.py) ... done
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11678 sha256=36720b8c3ab7e1b17392bd90efcdca002a900037edc1320481bd8ec03602536b
  Stored in directory: /root/.cache/pip/wheels/8f/ab/cb/45ccc40522d3a1c41e1d2ad53b8f33a62f394011ec38cd71c6
Successfully built wikipedia


In [230]:
from langchain_community.retrievers import WikipediaRetriever

In [231]:
#initialize retriever

retriever = WikipediaRetriever(
    top_k_results=2, lang='en'
)

# define your query
query = 'Geographical relationships b/n india & china'

#get realavant wikipedia docs
docs = retriever.invoke(query)

In [232]:
print(len(docs))

2


In [233]:
print(docs[0])

page_content='China and India maintained peaceful relations for thousands of years, but their relationship has varied since the Chinese Communist Party (CCP)'s victory in the Chinese Civil War in 1949 and the annexation of Tibet by the People's Republic of China. The two nations have sought economic cooperation with each other, while frequent border disputes and economic nationalism in both countries are major points of contention.
Cultural and economic relations between China and India date back to ancient times. The Silk Road not only served as a major trade route between India and China, but is also credited for facilitating the spread of Buddhism from India to East Asia. During the 19th century, China was involved in a growing opium trade with the East India Company, which exported opium grown in India. During World War II, both British India and the Republic of China (ROC) played a crucial role in halting the progress of Imperial Japan. After India became independent in 1947, it e

## **Vector Store Retriever**

In [234]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain.schema import Document

In [235]:
# Initialize the Google Generative AI embeddings model
google_embeddings = GoogleGenerativeAIEmbeddings(
    model='models/gemini-embedding-exp-03-07',  # Specify the model name
)

In [236]:
# create langchain documents

doc1 = Document(
    page_content = '''Mumbai Indians are the most successful IPL team with five championship titles.
Known for their fierce comebacks, they blend young talent with seasoned stars.
Rohit Sharma has been their iconic leader, shaping the team's winning mentality.
They have a massive fan base and a reputation for dominating under pressure.''',
    metadata = {"team":'Mumbai Indians'}
)

doc2 =  Document(
    page_content = '''CSK, led by the legendary MS Dhoni, are synonymous with consistency and resilience.
They have lifted the IPL trophy multiple times, earning respect across the league.
The team is famous for its “Dad’s Army” tag, with experienced players delivering big.
Their “Whistle Podu” fans create one of the most passionate atmospheres in cricket.''',
    metadata = {"team":'Chennai Super Kings'}
)

doc3 =   Document(
    page_content = '''RCB are known for star power, with legends like Virat Kohli, AB de Villiers, and Chris Gayle.
They’re famous for thrilling chases and explosive batting line-ups.
Despite never winning an IPL trophy, they have a loyal global fan following.
Their motto “Ee Sala Cup Namde” reflects their relentless pursuit of glory.''',
    metadata = {"team":'Royal Challengers Bengaluru'}
)

In [237]:
docs = [doc1, doc2, doc3]

In [238]:
from langchain_community.vectorstores import Chroma

vector_store = Chroma(
    embedding_function=google_embeddings,
    persist_directory='chroma_db',
    collection_name='IPLTeamDetails'
)

In [239]:
vector_store.add_documents(docs)

['a4948804-3846-4f49-aa7d-d743be24bee8',
 '717f8a4a-a1bc-42e1-8cb2-c624c05ee8d0',
 '31184790-dffb-4da5-93ce-4146b3f90064']

In [240]:
# convert vectore store as retriver
retriever = vector_store.as_retriever(search_kwargs={'k':1})

In [241]:
query = 'Who is the captain of RCB ?'
result = retriever.invoke(query)

In [242]:
print(result)

[Document(metadata={'team': 'Royal Challengers Bengaluru'}, page_content='RCB are known for star power, with legends like Virat Kohli, AB de Villiers, and Chris Gayle.\nThey’re famous for thrilling chases and explosive batting line-ups.\nDespite never winning an IPL trophy, they have a loyal global fan following.\nTheir motto “Ee Sala Cup Namde” reflects their relentless pursuit of glory.')]


## **MMR Retriever**

In [243]:
!pip install faiss-cpu

I0000 00:00:1755363631.428418      36 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 44.5 MB/s eta 0:00:00:00:0100:01


In [244]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain.schema import Document
from langchain_community.vectorstores  import FAISS

In [245]:
# Initialize the Google Generative AI embeddings model
google_embeddings = GoogleGenerativeAIEmbeddings(
    model='models/gemini-embedding-exp-03-07',  # Specify the model name
)

In [246]:
vectore_store = FAISS.from_documents(
    documents=docs,
    embedding=google_embeddings
)

In [247]:
vector_store.add_documents([doc3])

['28eca7b0-8cbf-425f-b514-68959618c03f']

In [248]:
# enable the retriever
retriever = vectore_store.as_retriever(
    search_type='mmr',
    search_kwargs={'k':2, 'lambda_mult':0.1} # lambda_mult (0-1)
)

In [249]:
query = 'Who is the captain of RCB ?'
results = retriever.invoke(query)
print(results)

[Document(metadata={'team': 'Royal Challengers Bengaluru'}, page_content='RCB are known for star power, with legends like Virat Kohli, AB de Villiers, and Chris Gayle.\nThey’re famous for thrilling chases and explosive batting line-ups.\nDespite never winning an IPL trophy, they have a loyal global fan following.\nTheir motto “Ee Sala Cup Namde” reflects their relentless pursuit of glory.'), Document(metadata={'team': 'Mumbai Indians'}, page_content="Mumbai Indians are the most successful IPL team with five championship titles.\nKnown for their fierce comebacks, they blend young talent with seasoned stars.\nRohit Sharma has been their iconic leader, shaping the team's winning mentality.\nThey have a massive fan base and a reputation for dominating under pressure.")]


## **Multi Query Retriever**

In [250]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain.schema import Document
from langchain_community.vectorstores  import FAISS


# Health topics
doc1 = Document(
    page_content = '''Regular exercise strengthens muscles, boosts cardiovascular health, and improves mood.
Even 30 minutes of daily physical activity can significantly reduce the risk of chronic diseases.
Activities like walking, swimming, or cycling are accessible and highly beneficial.''',
    metadata = {"topic": "Physical Fitness"}
)

doc2 = Document(
    page_content = '''Balanced nutrition is the cornerstone of a healthy lifestyle.
A diet rich in fruits, vegetables, whole grains, and lean proteins supports overall well-being.
Limiting processed foods and added sugars helps maintain energy levels and weight control.''',
    metadata = {"topic": "Healthy Diet"}
)

doc3 = Document(
    page_content = '''Adequate sleep is vital for mental clarity, emotional stability, and physical health.
Adults generally need 7–9 hours of quality sleep each night to function optimally.
Poor sleep patterns can increase the risk of heart disease, obesity, and depression.''',
    metadata = {"topic": "Sleep Health"}
)

doc4 = Document(
    page_content = '''Mental health is as important as physical health for a balanced life.
Practicing mindfulness, staying socially connected, and seeking help when needed are key strategies.
Reducing stress through hobbies, exercise, or meditation can improve overall well-being.''',
    metadata = {"topic": "Mental Health"}
)

doc5 = Document(
    page_content = '''Hydration plays a crucial role in regulating body temperature and joint lubrication.
Drinking enough water throughout the day supports digestion and cognitive function.
Dehydration can lead to fatigue, headaches, and impaired concentration.''',
    metadata = {"topic": "Hydration"}
)

# Other topics
doc6 = Document(
    page_content = '''The Eiffel Tower, an iconic symbol of Paris, was completed in 1889 for the World’s Fair.
Initially criticized, it has become one of the most visited monuments in the world.
Its sparkling light show attracts millions of visitors every year.''',
    metadata = {"topic": "Eiffel Tower"}
)

doc7 = Document(
    page_content = '''Artificial Intelligence is transforming industries from healthcare to finance.
It enables machines to learn, adapt, and make decisions with minimal human intervention.
Ethical considerations are key as AI becomes more integrated into daily life.''',
    metadata = {"topic": "Artificial Intelligence"}
)

doc8 = Document(
    page_content = '''The Amazon Rainforest is the largest tropical rainforest, spanning nine countries.
It plays a crucial role in regulating the Earth’s climate and producing oxygen.
Deforestation poses a significant threat to its biodiversity and ecological balance.''',
    metadata = {"topic": "Amazon Rainforest"}
)

doc9 = Document(
    page_content = '''Space exploration has expanded humanity’s understanding of the universe.
Missions to Mars and the Moon aim to uncover clues about the origins of life.
Technological advancements from space research benefit everyday life on Earth. Sun is full of energy''',
    metadata = {"topic": "Space Exploration"}
)

doc10 = Document(
    page_content = '''Blockchain technology enables secure, transparent, and decentralized transactions.
It underpins cryptocurrencies like Bitcoin and Ethereum but has applications beyond finance.
Sectors such as supply chain, healthcare, and voting systems are exploring its potential.''',
    metadata = {"topic": "Blockchain Technology"}
)


In [251]:
docs = [doc1, doc2, doc3, doc4, doc5, doc6, doc7, doc8, doc9, doc10]

In [252]:
# Initialize the Google Generative AI embeddings model
google_embeddings = GoogleGenerativeAIEmbeddings(
    model='models/gemini-embedding-exp-03-07',  # Specify the model name
)

In [253]:
vector_store = FAISS.from_documents(
    documents=docs,
    embedding=google_embeddings
)

In [254]:
# create a retriever 
similarity_retriever = vector_store.as_retriever(
    search_type='similarity',
    search_kwargs={'k':5}
)

In [255]:
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain.chat_models import init_chat_model

In [256]:
mult_retriever = MultiQueryRetriever.from_llm(
    retriever = vector_store.as_retriever(search_kwargs={'k':5}),
    llm = init_chat_model(model='gemini-2.0-flash', model_provider='google-genai')
)

In [257]:
# define query

query = 'how to maintain nutrition and energy?'

similary_results = similarity_retriever.invoke(query)
mult_results = mult_retriever.invoke(query)

In [258]:
for result in similary_results:
    print(result.page_content + '\n')

Balanced nutrition is the cornerstone of a healthy lifestyle.
A diet rich in fruits, vegetables, whole grains, and lean proteins supports overall well-being.
Limiting processed foods and added sugars helps maintain energy levels and weight control.

Hydration plays a crucial role in regulating body temperature and joint lubrication.
Drinking enough water throughout the day supports digestion and cognitive function.
Dehydration can lead to fatigue, headaches, and impaired concentration.

Regular exercise strengthens muscles, boosts cardiovascular health, and improves mood.
Even 30 minutes of daily physical activity can significantly reduce the risk of chronic diseases.
Activities like walking, swimming, or cycling are accessible and highly beneficial.

Mental health is as important as physical health for a balanced life.
Practicing mindfulness, staying socially connected, and seeking help when needed are key strategies.
Reducing stress through hobbies, exercise, or meditation can improv

In [259]:
for result in mult_results:
    print(result.page_content + '\n')

Balanced nutrition is the cornerstone of a healthy lifestyle.
A diet rich in fruits, vegetables, whole grains, and lean proteins supports overall well-being.
Limiting processed foods and added sugars helps maintain energy levels and weight control.

Hydration plays a crucial role in regulating body temperature and joint lubrication.
Drinking enough water throughout the day supports digestion and cognitive function.
Dehydration can lead to fatigue, headaches, and impaired concentration.

Space exploration has expanded humanity’s understanding of the universe.
Missions to Mars and the Moon aim to uncover clues about the origins of life.
Technological advancements from space research benefit everyday life on Earth. Sun is full of energy

Mental health is as important as physical health for a balanced life.
Practicing mindfulness, staying socially connected, and seeking help when needed are key strategies.
Reducing stress through hobbies, exercise, or meditation can improve overall well-be

## **Contextual Compression Retriever**

In [260]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain.schema import Document
from langchain_community.vectorstores  import FAISS


# Health topics
doc1 = Document(
    page_content = '''Regular exercise strengthens muscles, boosts cardiovascular health, and improves mood.
Even 30 minutes of daily physical activity can significantly reduce the risk of chronic diseases.
Activities like walking, swimming, or cycling are accessible and highly beneficial.''',
    metadata = {"topic": "Physical Fitness"}
)

doc2 = Document(
    page_content = '''Balanced nutrition is the cornerstone of a healthy lifestyle.
A diet rich in fruits, vegetables, whole grains, and lean proteins supports overall well-being.
Limiting processed foods and added sugars helps maintain energy levels and weight control.''',
    metadata = {"topic": "Healthy Diet"}
)

doc3 = Document(
    page_content = '''Adequate sleep is vital for mental clarity, emotional stability, and physical health.
Adults generally need 7–9 hours of quality sleep each night to function optimally.
Poor sleep patterns can increase the risk of heart disease, obesity, and depression.''',
    metadata = {"topic": "Sleep Health"}
)

doc4 = Document(
    page_content = '''Mental health is as important as physical health for a balanced life.
Practicing mindfulness, staying socially connected, and seeking help when needed are key strategies.
Reducing stress through hobbies, exercise, or meditation can improve overall well-being.''',
    metadata = {"topic": "Mental Health"}
)

doc5 = Document(
    page_content = '''Hydration plays a crucial role in regulating body temperature and joint lubrication.
Drinking enough water throughout the day supports digestion and cognitive function.
Dehydration can lead to fatigue, headaches, and impaired concentration.''',
    metadata = {"topic": "Hydration"}
)

# Other topics
doc6 = Document(
    page_content = '''The Eiffel Tower, an iconic symbol of Paris, was completed in 1889 for the World’s Fair.
Initially criticized, it has become one of the most visited monuments in the world.
Its sparkling light show attracts millions of visitors every year.''',
    metadata = {"topic": "Eiffel Tower"}
)

doc7 = Document(
    page_content = '''Artificial Intelligence is transforming industries from healthcare to finance.
It enables machines to learn, adapt, and make decisions with minimal human intervention.
Ethical considerations are key as AI becomes more integrated into daily life.''',
    metadata = {"topic": "Artificial Intelligence"}
)

doc8 = Document(
    page_content = '''The Amazon Rainforest is the largest tropical rainforest, spanning nine countries.
It plays a crucial role in regulating the Earth’s climate and producing oxygen.
Deforestation poses a significant threat to its biodiversity and ecological balance.''',
    metadata = {"topic": "Amazon Rainforest"}
)

doc9 = Document(
    page_content = '''Space exploration has expanded humanity’s understanding of the universe.
Missions to Mars and the Moon aim to uncover clues about the origins of life.
Technological advancements from space research benefit everyday life on Earth. Sun is full of energy''',
    metadata = {"topic": "Space Exploration"}
)

doc10 = Document(
    page_content = '''Blockchain technology enables secure, transparent, and decentralized transactions.
It underpins cryptocurrencies like Bitcoin and Ethereum but has applications beyond finance.
Sectors such as supply chain, healthcare, and voting systems are exploring its potential.''',
    metadata = {"topic": "Blockchain Technology"}
)


In [261]:
docs = [doc1, doc2, doc3, doc4, doc5, doc6, doc7, doc8, doc9, doc10]

In [262]:
# Initialize the Google Generative AI embeddings model
google_embeddings = GoogleGenerativeAIEmbeddings(
    model='models/gemini-embedding-exp-03-07',  # Specify the model name
)

In [263]:
vector_store = FAISS.from_documents(
    documents=docs,
    embedding=google_embeddings
)

In [264]:
# define base retriever
base_retriever = vector_store.as_retriever(search_kwargs={'k':2})

# define compressor using llm
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor

llm = init_chat_model(model='gemini-2.0-flash', model_provider='google-genai')
compressor = LLMChainExtractor.from_llm(llm)

In [265]:
# create contextual compression retriever
compression_retriever = ContextualCompressionRetriever(
    base_retriever = base_retriever,
    base_compressor = compressor
)

In [266]:
# define query
query = 'how to maintain nutrition and energy?'

results = compression_retriever.invoke(query)

In [267]:
print(results[0].page_content)

Limiting processed foods and added sugars helps maintain energy levels


# **Tools**

## **Built-in Tools**

### **Duck-Duck-GO Search**

In [268]:
!pip install ddgs

I0000 00:00:1755363659.096635      36 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 33.5 MB/s eta 0:00:0000:0100:01


In [269]:
from langchain_community.tools import DuckDuckGoSearchRun

In [ ]:
search_tool = DuckDuckGoSearchRun()

result = search_tool.invoke('india capital')
print(result)

### **Shell Tool**

In [271]:
from langchain_community.tools import ShellTool

shell_tool = ShellTool()

result = shell_tool.invoke('whoami')
print(result)

Executing command:
 whoami
root



/usr/local/lib/python3.11/dist-packages/langchain_community/tools/shell/tool.py:32: UserWarning: The shell tool has no safeguards by default. Use at your own risk.
  warnings.warn(


## **Custom Tools**

In [272]:
from langchain_core.tools import tool

In [273]:
# define tool function

@tool 
def Multiply(a:int, b:int):
    '''Multiply two numbers'''
    return a*b

In [274]:
result = Multiply.invoke({
    'a':5, 'b':10
})

print(result)

50


In [275]:
print(Multiply.name)
print(Multiply.description)
print(Multiply.args)

Multiply
Multiply two numbers
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [276]:
from langchain.tools import StructuredTool
from pydantic import BaseModel, Field

In [277]:
class MultiplyInput(BaseModel):
    a:int = Field(requirement=True, description='First number to mult')
    b:int = Field(requirement=True, description='Second number to mult')

In [278]:
def MultiplyFunction(a:int, b:int):
    return a*b

In [279]:
MultiplyTool = StructuredTool.from_function(
    func = MultiplyFunction,
    name='Multiply',
    description='Multiply two numbers',
    args_schema=MultiplyInput
)

In [280]:
result = MultiplyTool.invoke({
    'a':26, 'b':6
})
print(result)

156


In [281]:
from langchain.tools import BaseTool
from typing import Type

In [282]:
class MultiplyInput(BaseModel):
    a:int = Field(requirement=True, description='First number to mult')
    b:int = Field(requirement=True, description='Second number to mult')

In [283]:
class MultiplyTool(BaseTool):
    name:str = 'Multiply'
    description:str = 'multiply two numbers'
    args_schema: type[BaseModel] = MultiplyInput

    def _run(self, a:int, b:int) -> int:
        return a*b


In [284]:
multiply_tool = MultiplyTool()

result = multiply_tool.invoke({
    'a':14, 
    'b':15
})
print(result)

210


## **Tool-Kit**

In [285]:
from langchain_core.tools import tool

# custom tools

@tool
def add(a:int, b:int) -> int:
    '''Add two numbers'''
    return a+b

@tool
def Multiply(a:int, b:int) -> int:
    '''Multiply two numbers'''
    return a*b


In [286]:
class MathToolKit:
    def get_tools(self):
        return [add, Multiply]

In [287]:
tool_kit = MathToolKit()
tools = tool_kit.get_tools()

In [288]:
for tool in tools:
    print(tool.name)
    print(tool.description)
    print(tool.args)

add
Add two numbers
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}
Multiply
Multiply two numbers
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


## **Tool Calling**

In [289]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests
from langchain.chat_models import init_chat_model
from langchain_core.tools import InjectedToolArg
from typing import Annotated

In [290]:
# tool creation
@tool
def multiply(a:int , b:int) -> int:
    '''Multiply two numbers'''
    return a*b

In [291]:
# tool binding

model = init_chat_model(model='gemini-2.0-flash', model_provider='google-genai')

Agent = model.bind_tools([multiply])

In [292]:
result = Agent.invoke('what is 4 times 55')

In [293]:
# tool execution
tool_result = multiply.invoke(result.tool_calls[0])

In [294]:
query = HumanMessage('what is 4 times 55')
result = Agent.invoke('what is 4 times 55')
messages = [query, result, tool_result]

Agent.invoke(messages).content

'The result of 4 times 55 is 220.\n'

In [295]:
# currancy convertion tools
import json

@tool
def get_conversion_rate(base_currancy:str, target_currancy:str) -> float:
    '''This function fetches conversion factor between a base currancy and a target currancy'''
    url = f'https://v6.exchangerate-api.com/v6/{os.environ["ExchangeRateAPI"]}/pair/{base_currancy}/{target_currancy}'
    response = requests.get(url)
    response = response.json()
    return response['conversion_rate']

In [296]:
@tool
def convert(base_currancy_value:int, conversion_rate: float) -> float:
    '''Convert input currancy value to target currancy value'''
    
    return base_currancy_value * conversion_rate

In [297]:
response = convert.invoke({
    'base_currancy_value':500,
    'conversion_rate':87.5
})

print(response)

43750.0


In [298]:
response = get_conversion_rate.invoke({
    'base_currancy':'USD',
    'target_currancy':'INR'
})

print(response)

87.5396


## **Connect Tools with LLM**

In [299]:
messages = []

while True:
    
    input_query = input()

    if input_query in ['break', 'exit']:
        break
        
    query = HumanMessage(input_query)
    messages.append(query)
    response = agent.invoke(messages)
    messages.append(response)
    
    while response.tool_calls: # tool execution

        tool_call = response.tool_calls[0]
        
        if tool_call['name'] == 'get_conversion_rate':
            tool_message = get_conversion_rate.invoke(tool_call)
            messages.append(tool_message)

        elif tool_call['name'] == 'convert':
            tool_message = convert.invoke(tool_call)
            messages.append(tool_message)

        response = agent.invoke(messages)
        messages.append(response)
        
    print(response.content)
    

 break


In [300]:
for message in messages:
    print(message)
    print('\n')

In [301]:
from langchain.agents import create_react_agent, AgentExecutor
from langchain import hub
from langchain.chat_models import init_chat_model

In [302]:
import json

@tool
def get_weather(city: str ) -> str:
    """
    Get the current weather condition for a given city and country.
    """
    return "45 degree celcius"

response = get_weather.invoke({"city": "Delhi"})
print(response)


45 degree celcius


In [303]:
# pull react prompt langchain hub

prompt = hub.pull('hwchase17/react')

/usr/local/lib/python3.11/dist-packages/langsmith/client.py:241: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


In [304]:
# create initial agent
agent = create_react_agent(
    llm =  init_chat_model(model='gemini-2.5-flash', model_provider='google-genai'),
    tools = [get_weather],
    prompt = prompt
)

In [305]:
agent_executer = AgentExecutor(
    agent = agent,
    tools= [get_weather],
    verbose=True
)

In [306]:
response = agent_executer.invoke({
    "input":"Hello, what is the weather condition of patna?"
})

print(response)



> Entering new AgentExecutor chain...
Action: get_weather
Action Input: {"city": "patna", "country": "India"}45 degree celciusI now know the final answer
Final Answer: The weather condition of Patna is 45 degree Celsius.

> Finished chain.
{'input': 'Hello, what is the weather condition of patna?', 'output': 'The weather condition of Patna is 45 degree Celsius.'}


## **Thank You**